# Dalila — automated ASR experiment
Upload only the sanitized Advanced ZIP as a private Kaggle dataset; attach it to a notebook. Enable Internet. GPU optional; the default CPU run is closest to deployment. Read `reports/MODEL_AND_DATA_REVIEW.md` before setting the terms flag. No AFI records or API keys are needed. Public corpus is not independent of all model training.

In [ ]:
import os, sys, zipfile, pathlib, subprocess
from pathlib import Path
ZIP=next(iter(Path('/kaggle/input').rglob('Dalila_Language_Lab_Advanced.zip')),None)
assert ZIP, 'Attach the sanitized ZIP as a Kaggle dataset first'
with zipfile.ZipFile(ZIP) as z:
    assert all(not Path(i.filename).is_absolute() and '..' not in Path(i.filename).parts for i in z.infolist())
    z.extractall('/kaggle/working')
ROOT=Path('/kaggle/working/Dalila_Language_Lab_Advanced')
os.chdir(ROOT)
print(ROOT)


In [ ]:
# Change these only after reviewing the data/model terms.
ACCEPT_PUBLIC_TERMS=False
INCLUDE_LARGE=False
USE_CUDA=False
LIMIT=60
assert ACCEPT_PUBLIC_TERMS, 'Review terms and set ACCEPT_PUBLIC_TERMS=True for authorized research'
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt','-r','requirements-benchmark.txt'],check=True)
if INCLUDE_LARGE:
    subprocess.run([sys.executable,'-m','pip','install','-r','requirements-convert.txt'],check=True)


In [ ]:
models=['tn-small','tn-whisperx','small']+(['turbo','tn-codeswitch'] if INCLUDE_LARGE else [])
os.environ.update(ASR_DEVICE='cuda' if USE_CUDA else 'cpu',ASR_COMPUTE_TYPE='int8_float16' if USE_CUDA else 'int8',ASR_THREADS='4',ASR_BEAM_SIZE='3')
subprocess.run([sys.executable,'verify_models.py'],check=True)
subprocess.run([sys.executable,'prepare_models.py','--models',*models],check=True)
subprocess.run([sys.executable,'collect_public_audio.py','--limit',str(LIMIT),'--accept-public-terms'],check=True)
os.environ['ASR_OFFLINE']='1'
subprocess.run([sys.executable,'evaluate.py','--audio','data/public_linto/manifest.jsonl','--public','--models',*models,'--out','reports/public_asr.json'],check=True)


In [ ]:
import json, pandas as pd
r=json.load(open('reports/public_asr.json'))['results']
print(pd.DataFrame([{k:v for k,v in d.items() if k!='detail'}|{'model':m} for m,d in r.items()]))
# Export reports only; no model weights, recordings, secrets or user data.
with zipfile.ZipFile('/kaggle/working/dalila_benchmark_reports.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in Path('reports').glob('*'):
        if p.is_file():z.write(p,str(p))
print('/kaggle/working/dalila_benchmark_reports.zip')
